# 07 Multimodal Features: Phrase–Image Consistency

Builds fine-grained image–text features using the CLIP embeddings saved by `04`.

**Question:** does checking individual caption phrases against the image help detect manipulation beyond whole-caption similarity?

A caption can mention the correct subject but describe the wrong activity or setting. Each noun/verb phrase is embedded with the same frozen CLIP ViT-L/14 text encoder and compared with the cached whole-image embedding. No images are re-encoded and no image-only or text-only detector is replaced.

**Saved feature group:** `data/features/mm_phrase_<split>.parquet`, joined on `row_id` as in `03`/`04`. Requires matching processed tables from `02`, CLIP tables from `04`, and the spaCy English model. The normal run uses all `train_sample`, `validation` and `test` rows; test labels are not evaluated.

## 0. Setup

In [ ]:
import os
import sys
import time
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate, best_f1_threshold, combine_fake_score, detection_rate_by_type

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CACHE_DIR = PROJECT_ROOT / "data" / "interim" / "phrase_clip"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train_sample", "validation", "test"]
RANDOM_STATE = 4012
MAX_PHRASES = 8
CLIP_COLUMNS = ["clip_sim", "clip_sim_txt_norm", "clip_sim_img_norm", "clip_sim_txt_rank"]

sns.set_theme(style="whitegrid")


## 1. Phrase Feature Definitions

Extract up to eight unique noun chunks and eight verb-centred phrases per caption. Verb spans include dependency-linked modifiers, objects and prepositions, capped at 20 tokens.

For each category, summarise phrase–image cosine similarities using minimum, maximum, mean and standard deviation. `caption_gap` is whole-caption similarity minus mean phrase similarity. Counts and missing-category indicators are saved as controls, since caption structure alone can correlate with labels. Empty categories are zero-filled with `missing=1`.

The parser locates phrases; it does not predict whether the caption is fake. Label annotations are never used to select phrases.

In [ ]:
def phrases_from_doc(doc):
    nouns = list(dict.fromkeys(chunk.text for chunk in doc.noun_chunks))[:MAX_PHRASES]
    verbs = []
    for token in doc:
        if token.pos_ != "VERB":
            continue
        positions = [token.i]
        for child in token.children:
            if child.dep_ in {"aux", "auxpass", "neg", "prt", "advmod", "obj", "dobj", "attr", "prep"}:
                positions.extend(word.i for word in child.subtree)
        lo, hi = min(positions), max(positions) + 1
        if hi - lo <= 20:
            verbs.append(doc[lo:hi].text)
    return {"noun": nouns, "verb": list(dict.fromkeys(verbs))[:MAX_PHRASES]}


def build_phrase_features(rows, image_vectors, phrase_lookup, phrase_vectors):
    records = []
    for i, row in enumerate(rows.itertuples()):
        record = {"row_id": row.row_id}
        for category in ["noun", "verb"]:
            phrases = phrase_lookup[row.text][category]
            similarities = np.asarray([image_vectors[i] @ phrase_vectors[p] for p in phrases])
            prefix = f"mm_phrase_{category}_"
            record[prefix + "count"] = len(phrases)
            record[prefix + "missing"] = int(not phrases)
            stats = ({"min": similarities.min(), "max": similarities.max(),
                      "mean": similarities.mean(), "std": similarities.std()} if phrases else
                     dict.fromkeys(["min", "max", "mean", "std"], 0.))
            record.update({prefix + name: float(value) for name, value in stats.items()})
            record[prefix + "caption_gap"] = row.clip_sim - stats["mean"] if phrases else 0.
        records.append(record)
    return pd.DataFrame(records)


### Optional Function Check

This tiny check uses two-dimensional toy vectors only to check the feature calculation. It does not supply inputs to the notebook pipeline or write files. Run setup, the definitions above and this cell to test the function independently; delete this cell and heading when no longer needed. The normal workflow below always reads `02`/`04` outputs.

In [ ]:
toy_rows = pd.DataFrame({"row_id": ["toy_0", "toy_1"], "text": ["caption_0", "caption_1"], "clip_sim": [0.8, 0.2]})
toy_phrases = {text: {"noun": ["hospital"], "verb": []} for text in toy_rows.text}
toy_vectors = {"hospital": np.array([1., 0.])}
toy_features = build_phrase_features(toy_rows, np.eye(2), toy_phrases, toy_vectors)
assert toy_features.mm_phrase_noun_mean.tolist() == [1., 0.]
assert toy_features.mm_phrase_verb_missing.eq(1).all()
display(toy_features)


## 2. Load Processed Tables and CLIP Embeddings

Read matching row-keyed outputs from `02` and `04`. Join validation must preserve row count and order. Only a few stored caption vectors are retained as anchors for checking the text encoder's embedding space; image vectors are reused for every phrase comparison.

In [ ]:
data = {}
image_vectors = {}
caption_anchors = []

for split in SPLITS:
    rows = pd.read_parquet(PROCESSED_DIR / f"{split}.parquet")
    clip = pd.read_parquet(FEATURES_DIR / f"clip_{split}.parquet")
    emb = pd.read_parquet(FEATURES_DIR / f"clip_emb_{split}.parquet")
    assert rows.row_id.is_unique and clip.row_id.is_unique and emb.row_id.is_unique
    assert set(rows.row_id) == set(clip.row_id) == set(emb.row_id), f"row IDs differ in {split}"
    rows = rows.merge(clip, on="row_id", how="left", validate="one_to_one", sort=False)
    emb = emb.set_index("row_id").loc[rows.row_id]
    img_cols = sorted(c for c in emb if c.startswith("clip_img_"))
    txt_cols = sorted(c for c in emb if c.startswith("clip_txt_"))
    assert len(img_cols) == len(txt_cols) == 768, "Expected ViT-L/14 embeddings from 04"
    image = emb[img_cols].to_numpy(dtype=np.float32, copy=True)
    norms = np.linalg.norm(image, axis=1, keepdims=True)
    assert np.isfinite(image).all() and (norms > 0).all()
    image_vectors[split] = image / norms
    assert np.isfinite(rows[CLIP_COLUMNS].to_numpy()).all()
    for i in range(min(3, len(rows))):
        anchor = emb[txt_cols].iloc[i].to_numpy(dtype=np.float32, copy=True)
        assert np.isfinite(anchor).all() and np.linalg.norm(anchor) > 0
        caption_anchors.append((rows.text.iloc[i], anchor / np.linalg.norm(anchor)))
    data[split] = rows

assert not (set(data["train_sample"].id) & set(data["validation"].id)), "Article overlap"
all_rows = pd.concat(data.values(), ignore_index=True)
captions = pd.Series(all_rows.text.unique(), name="text")
pd.Series({split: len(df) for split, df in data.items()}, name="rows")


## 3. Caption Phrases

Compute phrases once per distinct caption. The cache key includes caption order, the phrase settings and the spaCy model/library version. Inspect the examples before interpreting results: some genuine caption details are not visible in the image.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "ner"])
phrase_config = {"captions": captions.tolist(), "max_phrases": MAX_PHRASES,
                 "spacy": spacy.__version__, "model": nlp.meta, "method": "noun_chunks_verb_subtrees_v1"}
phrase_key = hashlib.sha256(json.dumps(phrase_config, sort_keys=True).encode()).hexdigest()[:16]
phrase_path = CACHE_DIR / f"caption_phrases_{phrase_key}.json"

if phrase_path.exists():
    phrase_lookup = json.loads(phrase_path.read_text())
    print("Caption phrases loaded from cache")
else:
    start = time.time()
    phrase_lookup = {text: phrases_from_doc(doc)
                     for text, doc in zip(captions, nlp.pipe(captions, batch_size=128))}
    temporary = phrase_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(phrase_lookup))
    temporary.replace(phrase_path)
    print(f"spaCy phrases: {time.time() - start:.0f}s")

assert set(phrase_lookup) == set(captions)
pd.DataFrame([{"text": text, **phrase_lookup[text]} for text in captions.head(5)])


## 4. Phrase Embeddings

Use the same frozen ViT-L/14 text encoder as `04`. Only new text phrases are encoded; images are not reprocessed. Completed text chunks are cached separately, so an interrupted encoding run can reuse completed chunks.

Re-encode the caption anchors and check agreement with `04`'s stored caption vectors. This catches incompatible model spaces; matching row IDs alone do not prove that independently copied tables came from the same metadata version.

In [ ]:
import torch
import transformers
from src.clip_features import CLIP_NAME, best_device, load_clip, encode_texts

phrases = list(dict.fromkeys(phrase for item in phrase_lookup.values()
                            for category in ["noun", "verb"] for phrase in item[category]))
prompts = list(dict.fromkeys(phrases + [text for text, _ in caption_anchors]))
device = best_device()
model, _, tokenizer = load_clip(device, CLIP_NAME)
encoder_config = {"model": CLIP_NAME, "revision": getattr(model.config, "_commit_hash", None),
                  "torch": str(torch.__version__), "transformers": transformers.__version__,
                  "device": device, "batch_size": 64, "prompts": prompts}
encoder_key = hashlib.sha256(json.dumps(encoder_config, sort_keys=True).encode()).hexdigest()[:16]
embedding_dir = CACHE_DIR / encoder_key
embedding_dir.mkdir(parents=True, exist_ok=True)

parts = []
start = time.time()
for lo in range(0, len(prompts), 1024):
    batch = prompts[lo:lo + 1024]
    path = embedding_dir / f"phrases_{lo:07d}.npy"
    if path.exists():
        part = np.load(path, allow_pickle=False)
    else:
        part = encode_texts(model, tokenizer, batch, device, batch_size=64)
        temporary = path.with_suffix(".tmp")
        with temporary.open("wb") as stream:
            np.save(stream, part, allow_pickle=False)
        temporary.replace(path)
    assert part.shape == (len(batch), 768) and np.isfinite(part).all()
    parts.append(part)
    print(f"{min(lo + 1024, len(prompts)):,}/{len(prompts):,} prompts; {time.time() - start:.0f}s", flush=True)

prompt_emb = np.concatenate(parts)
phrase_vectors = dict(zip(prompts, prompt_emb))
for text, expected in caption_anchors:
    assert np.allclose(phrase_vectors[text], expected, atol=5e-4, rtol=0), "Text encoder differs from 04"
print(f"{len(phrases):,} distinct phrases; {CLIP_NAME} on {device}")
del model


## 5. Phrase–Image Consistency

For each pair, compare its caption phrases with its image embedding. A low minimum score can indicate an unsupported detail; variation measures disagreement between caption parts. `caption_gap` asks whether the whole caption matches better than its individual phrases.

These are hypotheses, not proof of manipulation. Invisible context, identity recognition and phrase length can all affect similarity. Counts/missing indicators are evaluated separately as a structure control.

In [ ]:
features = pd.concat([build_phrase_features(rows, image_vectors[split], phrase_lookup, phrase_vectors)
                      for split, rows in data.items()], ignore_index=True)
assert features.row_id.is_unique
assert np.isfinite(features.drop(columns="row_id").to_numpy()).all()
feature_columns = [c for c in features if c != "row_id"]
print(f"{len(feature_columns)} features for {len(features):,} image-caption pairs")
features[feature_columns].describe().round(3)


## 6. Save Features

Write one `mm_phrase` table per split, following the same `row_id` convention as `03`/`04`. Labels, article IDs, image paths and genuine reference captions are not included as feature columns. Test features use exactly the same fixed calculation; test labels are not evaluated.

In [ ]:
def save_features(features, group):
    for split, rows in data.items():
        part = rows[["row_id"]].merge(features, on="row_id", how="left", validate="one_to_one", sort=False)
        assert part.row_id.tolist() == rows.row_id.tolist()
        assert np.isfinite(part.drop(columns="row_id").to_numpy()).all()
        path = FEATURES_DIR / f"{group}_{split}.parquet"
        temporary = path.with_suffix(".tmp.parquet")
        part.to_parquet(temporary, index=False)
        temporary.replace(path)
        print(f"{path.relative_to(PROJECT_ROOT)}  {part.shape}")


save_features(features, "mm_phrase")


## 7. Validation Comparison

Use the same scaled logistic regression and rows for three experiments:

- **CLIP:** existing whole-caption consistency features from `04`.
- **CLIP + structure:** add only phrase counts/missing indicators.
- **CLIP + phrase consistency:** add the full new group.

Fit separately for `image_fake` and `text_fake`; combine outputs using the existing evaluation helper. Thresholds are selected on validation. This is a controlled feature test, not a replacement for `05`'s full model. Inspect manipulation-type results and retain negative findings. Repeated validation comparisons are exploratory; no test evaluation is performed here.

In [ ]:
joined = {split: rows.merge(features, on="row_id", how="left", validate="one_to_one")
          for split, rows in data.items()}
train, val = joined["train_sample"], joined["validation"]
structure_columns = [c for c in feature_columns if c.endswith(("_count", "_missing"))]
experiments = {
    "CLIP": CLIP_COLUMNS,
    "CLIP + structure": CLIP_COLUMNS + structure_columns,
    "CLIP + phrase consistency": CLIP_COLUMNS + feature_columns,
}

results, by_type = [], []
for name, columns in experiments.items():
    predictions = {}
    for target in ["image_fake", "text_fake"]:
        assert train[target].nunique() == val[target].nunique() == 2
        classifier = make_pipeline(StandardScaler(), LogisticRegression(C=1., max_iter=2000, random_state=RANDOM_STATE))
        classifier.fit(train[columns], train[target])
        predictions[target] = classifier.predict_proba(val[columns])[:, 1]
    scores = {**predictions, "is_fake": combine_fake_score(predictions["image_fake"], predictions["text_fake"])}
    thresholds = {target: best_f1_threshold(val[target], score) for target, score in scores.items()}
    results.append(evaluate(val, predictions["image_fake"], predictions["text_fake"], thresholds)
                   .reset_index(names="target").assign(experiment=name))
    by_type.append(detection_rate_by_type(val, scores["is_fake"], thresholds["is_fake"])
                   .reset_index().assign(experiment=name))

results = pd.concat(results, ignore_index=True)
by_type = pd.concat(by_type, ignore_index=True)
display(results.round(3))
display(by_type.round(3))


## 8. Integration with 05

Two changes are required to use this group in `05`:

1. Add `"mm_phrase"` to `FEATURE_GROUPS`, so its saved Parquet files are joined on `row_id`. Check unique keys, complete coverage and no missing new features.
2. Include the `mm_phrase_*` columns in level-2 feature selection:

```python
CROSS = [c for c in train.columns
         if c.startswith(("clip_sim", "clip_face_", "mm_phrase_"))]
ALL = TEXT + IMAGE + CROSS + L1
```

Compare against the original CROSS list with identical rows/model settings and validation protocol. This notebook does not modify `05` automatically. A later, separate extension can compare person-name prompts with face embeddings; it is not implemented here.

# Summary

**Saved feature group** (`data/features/<group>_<split>.parquet`, joined on `row_id`):

| Group | Columns | What it targets |
|---|---|---|
| `mm_phrase` | 14 noun/verb similarity summaries, whole-caption gaps and availability controls | Image–caption inconsistencies, particularly caption swaps |

**Findings**
- ✏️[FILL: whether phrase consistency improves on global CLIP similarity and the structure control]
- ✏️[FILL: manipulation types helped or harmed; false-positive examples involving invisible context]
- ✏️[FILL: extraction runtime and whether the group should be retained in the full model]

Use training data for fitting, validation for comparisons/thresholds, and test only after the final model configuration is fixed. No measured improvement is claimed until this notebook is run on the real inputs.